In [1]:
from XRootD.client import FileSystem
import json
from tqdm import tqdm

# Define the sample nick for which the filelist information needs to be fixed
SAMPLE_NICK = "DYto2Mu-2Jets_Bin-2J-MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8_RunIII2024Summer24NanoAODv15-150X"

In [2]:
%%capture cap_sample_database

# Get sample metadata from the sample database
jq_string = f"' .[] | select(.nick | startswith(\"{SAMPLE_NICK}\")) '"
!cat ../nanoAOD_v15/datasets.json | jq -r -M {jq_string}

In [3]:
# Load the sample information into a dictionary
sample_info = json.loads(cap_sample_database.stdout)

In [4]:
%%capture cap_dasgoclient

# Query files of the /DYto2Mu-2Jets_Bin-2J-MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8/RunIII2024Summer24NanoAODv15-150X_mcRun3_2024_realistic_v2-v3/NANOAODSIM dataset and extract file address and number of events
!dasgoclient --query="file dataset=/DYto2Mu-2Jets_Bin-2J-MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8/RunIII2024Summer24NanoAODv15-150X_mcRun3_2024_realistic_v2-v3/NANOAODSIM" -json

In [5]:
# Load the captured output of the dasgoclient command and compile the filelist
files = [
    {
        "name": entry["file"][0]["name"],
        "nevents": entry["file"][0]["nevents"],
        "status": None,
        "stat_info": None
    }
    for entry in json.loads(cap_dasgoclient.stdout)
]

In [7]:
# Create the GridKA file system client
gridka_fs = FileSystem("root://cmsdcache-kit-disk.gridka.de:1094")

for file in tqdm(files, desc="Checking files on GridKA", unit="file", total=len(files)):
    # Check if the ROOT file is on GridKA
    status, stat_info = gridka_fs.stat(file["name"])

    # Add information to file dict
    file["status"] = status
    file["stat_info"] = stat_info

# Build list of available files
available_files = list(sorted((file for file in files if file["status"].code == 0), key=lambda x: x["name"]))

# Print summary of available files and events
nevents = sum(file["nevents"] for file in files)
nfiles = len(files)
available_nevents = sum(file["nevents"] for file in available_files)
available_nfiles = len(available_files)
print(f"Summary for {SAMPLE_NICK}:")
print(f"    Available files: {available_nfiles}/{nfiles}")
print(f"    Available events: {available_nevents}/{nevents}")

Checking files on GridKA:   0%|          | 0/1739 [00:00<?, ?file/s]

Checking files on GridKA: 100%|██████████| 1739/1739 [00:09<00:00, 187.81file/s]

Summary for DYto2Mu-2Jets_Bin-2J-MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8_RunIII2024Summer24NanoAODv15-150X:
    Available files: 1674/1739
    Available events: 276486780/285910646


In [8]:
# Update information in the sample database
with open("../nanoAOD_v15/datasets.json", "r") as f:
    datasets = json.load(f)
datasets[SAMPLE_NICK]["nevents"] = available_nevents
datasets[SAMPLE_NICK]["nfiles"] = available_nfiles
with open("../nanoAOD_v15/datasets.json", "w") as f:
    json.dump(datasets, f, indent=4, sort_keys=True)

In [9]:
# Update information in the filelist file 
sample_file = f"../nanoAOD_v15/{sample_info['era']}/{sample_info['sample_type']}/{SAMPLE_NICK}.json"
with open(sample_file, "r") as f:
    dataset = json.load(f)
dataset["filelist"] = ["root://xrootd-cms.infn.it//" + file["name"] for file in available_files]
dataset["nevents"] = available_nevents
dataset["nfiles"] = available_nfiles
with open(sample_file, "w") as f:
    json.dump(dataset, f, indent=4, sort_keys=True)

../nanoAOD_v15/2024/dyjets_amcatnlo/DYto2Mu-2Jets_Bin-2J-MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8_RunIII2024Summer24NanoAODv15-150X.json
